# Notebook 05 — Ablation du seuil de décision

## Objectif

Évaluer la sensibilité des performances au seuil de décision, sans réentraîner les modèles.

Pour chaque expérience du Notebook 04, deux seuils sont comparés :

- **Seuil sélectionné sur la validation interne** : seuil sauvegardé dans le fichier `.npz` et utilisé dans l'évaluation principale.
- **Seuil fixe 0,5** : seuil de référence appliqué aux mêmes scores sauvegardés.

Les deux seuils sont appliqués aux mêmes prédictions intra-dataset et cross-dataset. Les scores continus ne changent pas : la PR-AUC et la ROC-AUC doivent donc rester identiques. Seules les métriques dépendant du seuil changent.

**Aucune donnée de test ni donnée cible n'est utilisée pour choisir le seuil 0,5.** Le seuil 0,5 est fixé à l'avance comme analyse de sensibilité.

## Question étudiée

Dans quelle mesure les conclusions sur le F1, la précision, le rappel et le taux d'alertes dépendent-elles du seuil sélectionné sur la validation interne ?

## Précautions d'interprétation

- Le seuil sélectionné est appris uniquement sur la validation interne du dataset source.
- Le seuil 0,5 est un point de comparaison fixe, pas nécessairement un seuil optimal.
- Aucun seuil oracle calculé sur les labels du test ou de la cible n'est utilisé comme performance officielle.
- Les 15 évaluations cross d'une même direction réutilisent les mêmes observations cibles. Les écarts-types décrivent la variabilité des expériences, mais ne constituent pas des intervalles de confiance fondés sur des observations indépendantes.

## 1. Imports et configuration

Le notebook doit être exécuté depuis le dossier `notebooks/` ou depuis la racine du projet. Les chemins sont construits relativement à la racine détectée.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

TABLES_DIR = ROOT_DIR / "results" / "tables"
PRED_DIR = ROOT_DIR / "results" / "predictions" / "cross_dataset_v1_1"

RESULTS_FILE = TABLES_DIR / "cross_dataset_baselines_v1_1.csv"

SEEDS = [42, 123, 2026]
FOLDS = range(1, 6)
MODELS = ["Logistic Regression", "Random Forest", "MLP"]
DIRECTIONS = [
    ("CIC-IDS2017", "UNSW-NB15", "CIC", "UNSW"),
    ("UNSW-NB15", "CIC-IDS2017", "UNSW", "CIC"),
]

THRESHOLD_FIXED = 0.5

print("Racine du projet :", ROOT_DIR)
print("CSV principal   :", RESULTS_FILE)
print("Prédictions     :", PRED_DIR)
print("Seuil fixe      :", THRESHOLD_FIXED)

assert RESULTS_FILE.exists(), f"CSV principal introuvable : {RESULTS_FILE}"
assert PRED_DIR.exists(), f"Dossier de prédictions introuvable : {PRED_DIR}"

Racine du projet : D:\Master-SDIA\CYB-1-IDS-Generalization
CSV principal   : D:\Master-SDIA\CYB-1-IDS-Generalization\results\tables\cross_dataset_baselines_v1_1.csv
Prédictions     : D:\Master-SDIA\CYB-1-IDS-Generalization\results\predictions\cross_dataset_v1_1
Seuil fixe      : 0.5


## 2. Chargement et contrôle des résultats du Notebook 04

Le CSV du Notebook 04 fournit les clés des expériences et les métriques de référence. Les prédictions sauvegardées permettent de recalculer les métriques avec les deux seuils.

In [2]:
results = pd.read_csv(RESULTS_FILE)

KEYS = ["source", "target", "model", "fold", "seed"]
required_columns = set(KEYS + [
    "threshold", "intra_f1", "cross_f1",
    "intra_pr_auc", "cross_pr_auc",
    "intra_roc_auc", "cross_roc_auc"
])

missing = required_columns - set(results.columns)
if missing:
    raise ValueError(f"Colonnes absentes du CSV du Notebook 04 : {sorted(missing)}")

if results.duplicated(KEYS).any():
    raise ValueError("Le CSV contient des expériences dupliquées.")

expected_count = len(DIRECTIONS) * len(MODELS) * len(list(FOLDS)) * len(SEEDS)
print("Nombre d'expériences :", len(results), "/", expected_count)

if len(results) != expected_count:
    raise ValueError(
        f"Nombre d'expériences inattendu : {len(results)} au lieu de {expected_count}. "
        "Vérifier la complétude du Notebook 04."
    )

display(
    results.groupby(["source", "target", "model"])
    .size()
    .rename("nombre_experiences")
    .reset_index()
)

Nombre d'expériences : 90 / 90


,source,target,model,nombre_experiences
0,CIC-IDS2017,UNSW-NB15,Logistic Regression,15
1,CIC-IDS2017,UNSW-NB15,MLP,15
2,CIC-IDS2017,UNSW-NB15,Random Forest,15
3,UNSW-NB15,CIC-IDS2017,Logistic Regression,15
4,UNSW-NB15,CIC-IDS2017,MLP,15
5,UNSW-NB15,CIC-IDS2017,Random Forest,15


## 3. Fonctions de calcul

Les métriques au seuil sont recalculées à partir des scores sauvegardés. La PR-AUC et la ROC-AUC sont également recalculées pour contrôler qu'elles ne dépendent pas du seuil.

In [3]:
def prediction_filename(source_short, target_short, model_name, fold, seed):
    model_tag = model_name.replace(" ", "_")
    return f"{source_short}_to_{target_short}_{model_tag}_f{fold}_s{seed}.npz"


def threshold_metrics(y_true, scores, threshold):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)

    if len(y_true) != len(scores):
        raise ValueError("Les labels et les scores n'ont pas la même longueur.")
    if not np.isfinite(scores).all():
        raise ValueError("Les scores contiennent des valeurs non finies.")
    if not 0 <= float(threshold) <= 1:
        raise ValueError(f"Seuil invalide : {threshold}")

    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "threshold": float(threshold),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "prevalence": float(np.mean(y_true)),
        "alert_rate": float(np.mean(y_pred)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "pr_auc": average_precision_score(y_true, scores),
        "roc_auc": roc_auc_score(y_true, scores),
    }

## 4. Recalcul des métriques pour les deux seuils

Pour chaque expérience, le seuil sélectionné est lu directement dans le `.npz`. Il est comparé au seuil fixe 0,5 sur les mêmes prédictions source et cible.

Le F1 de validation interne est aussi recalculé avec les scores de validation sauvegardés, afin de contrôler le seuil sélectionné.

In [4]:
rows = []
missing_prediction_files = []

for r in results.itertuples(index=False):
    source_short = "CIC" if r.source == "CIC-IDS2017" else "UNSW"
    target_short = "CIC" if r.target == "CIC-IDS2017" else "UNSW"

    filename = prediction_filename(
        source_short, target_short, r.model, int(r.fold), int(r.seed)
    )
    filepath = PRED_DIR / filename

    if not filepath.exists():
        missing_prediction_files.append(str(filepath))
        continue

    with np.load(filepath) as pred:
        required_npz = {
            "threshold",
            "source_test_y_true", "source_test_score",
            "target_y_true", "target_score",
            "inner_val_y_true", "inner_val_score"
        }
        missing_npz = required_npz - set(pred.files)
        if missing_npz:
            raise KeyError(f"{filename}: clés NPZ absentes {sorted(missing_npz)}")

        selected_threshold = float(pred["threshold"])

        y_intra = pred["source_test_y_true"].astype(int)
        s_intra = pred["source_test_score"].astype(float)

        y_cross = pred["target_y_true"].astype(int)
        s_cross = pred["target_score"].astype(float)

        y_val = pred["inner_val_y_true"].astype(int)
        s_val = pred["inner_val_score"].astype(float)

    # Contrôle du seuil sauvegardé contre le CSV de référence
    if not np.isclose(selected_threshold, float(r.threshold), atol=1e-12):
        raise AssertionError(
            f"Seuil différent entre CSV et NPZ : {filename} "
            f"({r.threshold} vs {selected_threshold})"
        )

    # Validation interne : contrôle du seuil sélectionné
    val_selected = threshold_metrics(y_val, s_val, selected_threshold)
    val_fixed = threshold_metrics(y_val, s_val, THRESHOLD_FIXED)

    # Évaluation intra et cross avec les deux seuils
    for evaluation_name, y, scores in [
        ("intra", y_intra, s_intra),
        ("cross", y_cross, s_cross)
    ]:
        selected = threshold_metrics(y, scores, selected_threshold)
        fixed = threshold_metrics(y, scores, THRESHOLD_FIXED)

        # Vérification du recalcul du F1 principal du Notebook 04
        reference_f1 = float(getattr(r, f"{evaluation_name}_f1"))
        if not np.isclose(selected["f1"], reference_f1, atol=1e-10):
            raise AssertionError(
                f"F1 recalculé différent du Notebook 04 : {filename}, "
                f"{evaluation_name}, {selected['f1']} vs {reference_f1}"
            )

        rows.append({
            "source": r.source,
            "target": r.target,
            "model": r.model,
            "fold": int(r.fold),
            "seed": int(r.seed),
            "evaluation": evaluation_name,
            "selected_threshold": selected_threshold,
            "fixed_threshold": THRESHOLD_FIXED,
            **{f"selected_{k}": v for k, v in selected.items() if k != "threshold"},
            **{f"fixed_{k}": v for k, v in fixed.items() if k != "threshold"},
            "delta_f1_fixed_minus_selected": fixed["f1"] - selected["f1"],
            "delta_precision_fixed_minus_selected": fixed["precision"] - selected["precision"],
            "delta_recall_fixed_minus_selected": fixed["recall"] - selected["recall"],
            "delta_alert_rate_fixed_minus_selected": fixed["alert_rate"] - selected["alert_rate"],
            "validation_f1_selected_threshold": val_selected["f1"],
            "validation_f1_fixed_05": val_fixed["f1"],
        })

if missing_prediction_files:
    raise FileNotFoundError(
        f"{len(missing_prediction_files)} fichiers de prédictions sont absents. "
        f"Exemple : {missing_prediction_files[0]}"
    )

ablation = pd.DataFrame(rows)

print("Lignes calculées :", len(ablation))
print("Attendu           :", len(results) * 2)
print("Contrôles F1 du Notebook 04 : réussis")

display(ablation.head())

Lignes calculées : 180
Attendu           : 180
Contrôles F1 du Notebook 04 : réussis


,source,target,model,fold,seed,evaluation,selected_threshold,fixed_threshold,selected_f1,selected_precision,selected_recall,selected_accuracy,selected_balanced_accuracy,selected_prevalence,selected_alert_rate,selected_tn,selected_fp,selected_fn,selected_tp,selected_pr_auc,selected_roc_auc,fixed_f1,fixed_precision,fixed_recall,fixed_accuracy,fixed_balanced_accuracy,fixed_prevalence,fixed_alert_rate,fixed_tn,fixed_fp,fixed_fn,fixed_tp,fixed_pr_auc,fixed_roc_auc,delta_f1_fixed_minus_selected,delta_precision_fixed_minus_selected,delta_recall_fixed_minus_selected,delta_alert_rate_fixed_minus_selected,validation_f1_selected_threshold,validation_f1_fixed_05
0,CIC-IDS2017,UNSW-NB15,Logistic Regression,1,42,intra,0.473594,0.5,0.530716,0.444161,0.659171,0.769312,0.727828,0.19789,0.293685,12778,3265,1349,2609,0.483682,0.815382,0.408224,0.492330,0.348661,0.79996,0.629981,0.19789,0.140143,14620,1423,2578,1380,0.483682,0.815382,-1.224924e-01,0.048169,-0.310510,-0.153542,0.631799,0.396878
1,CIC-IDS2017,UNSW-NB15,Logistic Regression,1,42,cross,0.473594,0.5,0.000467,0.011236,0.000239,0.871620,0.498609,0.12577,0.002670,87159,264,12574,3,0.647029,0.866635,0.000467,0.011407,0.000239,0.87166,0.498632,0.12577,0.002630,87163,260,12574,3,0.647029,0.866635,1.455278e-07,0.000171,0.000000,-0.000040,0.631799,0.396878
2,CIC-IDS2017,UNSW-NB15,Logistic Regression,1,123,intra,0.476348,0.5,0.533837,0.447851,0.660687,0.771661,0.729864,0.19789,0.291935,12819,3224,1343,2615,0.487779,0.818600,0.408599,0.485069,0.352956,0.79781,0.630258,0.19789,0.143993,14560,1483,2561,1397,0.487779,0.818600,-1.252379e-01,0.037219,-0.307731,-0.147943,0.622754,0.394670
3,CIC-IDS2017,UNSW-NB15,Logistic Regression,1,123,cross,0.476348,0.5,0.000467,0.011278,0.000239,0.871630,0.498615,0.12577,0.002660,87160,263,12574,3,0.646955,0.866638,0.000467,0.011407,0.000239,0.87166,0.498632,0.12577,0.002630,87163,260,12574,3,0.646955,0.866638,1.091543e-07,0.000129,0.000000,-0.000030,0.622754,0.394670
4,CIC-IDS2017,UNSW-NB15,Logistic Regression,1,2026,intra,0.472706,0.5,0.598608,0.469414,0.825922,0.780811,0.797802,0.19789,0.348183,12348,3695,689,3269,0.491706,0.824893,0.409486,0.490993,0.351187,0.79956,0.630683,0.19789,0.141543,14602,1441,2568,1390,0.491706,0.824893,-1.891224e-01,0.021578,-0.474735,-0.206640,0.649515,0.411657


## 5. Contrôle : les métriques de classement restent inchangées

Changer le seuil transforme les scores continus en classes prédites, mais ne modifie pas les scores eux-mêmes. La PR-AUC et la ROC-AUC doivent donc être identiques entre les deux seuils.

In [5]:
max_pr_auc_diff = (
    ablation["selected_pr_auc"] - ablation["fixed_pr_auc"]
).abs().max()

max_roc_auc_diff = (
    ablation["selected_roc_auc"] - ablation["fixed_roc_auc"]
).abs().max()

print("Écart maximal PR-AUC :", max_pr_auc_diff)
print("Écart maximal ROC-AUC :", max_roc_auc_diff)

assert max_pr_auc_diff < 1e-12
assert max_roc_auc_diff < 1e-12

print("Contrôle réussi : PR-AUC et ROC-AUC sont indépendantes du seuil.")

Écart maximal PR-AUC : 0.0
Écart maximal ROC-AUC : 0.0
Contrôle réussi : PR-AUC et ROC-AUC sont indépendantes du seuil.


## 6. Résumé comparatif par direction, modèle et type d'évaluation

Les valeurs sont résumées par moyenne et écart-type sur les 15 expériences de chaque combinaison direction-modèle-évaluation. Les écarts-types sont descriptifs et ne doivent pas être interprétés comme des intervalles de confiance.

In [6]:
METRICS_TO_SUMMARIZE = [
    "selected_threshold",
    "selected_f1", "fixed_f1",
    "selected_precision", "fixed_precision",
    "selected_recall", "fixed_recall",
    "selected_alert_rate", "fixed_alert_rate",
    "delta_f1_fixed_minus_selected",
    "delta_precision_fixed_minus_selected",
    "delta_recall_fixed_minus_selected",
    "delta_alert_rate_fixed_minus_selected",
    "validation_f1_selected_threshold",
    "validation_f1_fixed_05",
]

summary = (
    ablation
    .groupby(["source", "target", "model", "evaluation"])[METRICS_TO_SUMMARIZE]
    .agg(["mean", "std"])
    .round(4)
)

display(summary)

selected_threshold         selected_f1         fixed_f1         selected_precision         fixed_precision         selected_recall         fixed_recall          \
                                                                     mean     std        mean     std     mean     std               mean     std            mean     std            mean     std         mean     std   
source      target      model               evaluation                                                                                                                                                                   
CIC-IDS2017 UNSW-NB15   Logistic Regression cross                  0.4742  0.0029      0.0533  0.2047   0.0006  0.0005             0.0725  0.2372          0.0133  0.0075          0.0463  0.1785       0.0003  0.0002   
                                            intra                  0.4742  0.0029      0.6062  0.0350   0.4124  0.0398             0.4937  0.0249          0.4912  0.0145          0.7871  0.0674       0.3607  0.0779   
                        MLP                 cross                  0.3125  0.1153      0.0038  0.0038   0.0029  0.0038             0.2448  0.2461          0.3640  0.3237          0.0020  0.0020       0.0015  0.0019   
                                            intra                  0.3125  0.1153      0.7693  0.0337   0.6854  0.0228             0.6972  0.0977          0.9138  0.1122          0.8841  0.1016       0.5685  0.1166   
                        Random Forest       cross                  0.6889  0.0517      0.0000  0.0000   0.0000  0.0000             0.0000  0.0000          0.0000  0.0000          0.0000  0.0000       0.0000  0.0000   
                                            intra                  0.6889  0.0517      0.9536  0.0237   0.9513  0.0227             0.9402  0.0453          0.9284  0.0439          0.9687  0.0046       0.9765  0.0030   
UNSW-NB15   CIC-IDS2017 Logistic Regression cross                  0.6951  0.0185      0.4474  0.0139   0.3321  0.0008             0.3506  0.0616          0.1995  0.0006          0.7109  0.2228       0.9885  0.0008   
                                            intra                  0.6951  0.0185      0.4261  0.2216   0.3642  0.0204             0.3705  0.1555          0.2305  0.0121          0.5170  0.3308       0.8687  0.0633   
                        MLP                 cross                  0.7381  0.2608      0.1385  0.0571   0.1611  0.0134             0.1526  0.0631          0.1551  0.0210          0.1317  0.0602       0.1696  0.0137   
                                            intra                  0.7381  0.2608      0.6066  0.3365   0.9133  0.0353             0.8831  0.1355          0.9009  0.0198          0.5558  0.3896       0.9299  0.0739   
                        Random Forest       cross                  0.7257  0.3261      0.1008  0.1349   0.1118  0.0498             0.0754  0.0936          0.1093  0.0449          0.1693  0.2316       0.1163  0.0569   
                                            intra                  0.7257  0.3261      0.5732  0.3822   0.9532  0.0117             0.9676  0.0277          0.9485  0.0073          0.5246  0.4229       0.9581  0.0210   

                                                       selected_alert_rate         fixed_alert_rate         delta_f1_fixed_minus_selected         delta_precision_fixed_minus_selected          \
                                                                      mean     std             mean     std                          mean     std                                 mean     std   
source      target      model               evaluation                                                                                                                                           
CIC-IDS2017 UNSW-NB15   Logistic Regression cross                   0.0087  0.0235           0.0027  0.0003                       -0.0527  0.2042                              -0.0592  0.2297   
        

## 7. Résumé ciblé du F1 et du taux d'alertes

Ce tableau compact facilite la lecture des effets du seuil fixe 0,5 par rapport au seuil sélectionné sur la validation interne.

In [7]:
compact = (
    ablation
    .groupby(["source", "target", "model", "evaluation"])
    .agg(
        seuil_selectionne_moyen=("selected_threshold", "mean"),
        f1_seuil_selectionne_moyen=("selected_f1", "mean"),
        f1_seuil_05_moyen=("fixed_f1", "mean"),
        delta_f1_moyen=("delta_f1_fixed_minus_selected", "mean"),
        precision_selectionnee=("selected_precision", "mean"),
        precision_05=("fixed_precision", "mean"),
        rappel_selectionne=("selected_recall", "mean"),
        rappel_05=("fixed_recall", "mean"),
        alertes_selectionnees=("selected_alert_rate", "mean"),
        alertes_05=("fixed_alert_rate", "mean"),
    )
    .round(4)
    .reset_index()
)

display(compact)

,source,target,model,evaluation,seuil_selectionne_moyen,f1_seuil_selectionne_moyen,f1_seuil_05_moyen,delta_f1_moyen,precision_selectionnee,precision_05,rappel_selectionne,rappel_05,alertes_selectionnees,alertes_05
0,CIC-IDS2017,UNSW-NB15,Logistic Regression,cross,0.4742,0.0533,0.0006,-0.0527,0.0725,0.0133,0.0463,0.0003,0.0087,0.0027
1,CIC-IDS2017,UNSW-NB15,Logistic Regression,intra,0.4742,0.6062,0.4124,-0.1937,0.4937,0.4912,0.7871,0.3607,0.3155,0.1455
2,CIC-IDS2017,UNSW-NB15,MLP,cross,0.3125,0.0038,0.0029,-0.0009,0.2448,0.3640,0.0020,0.0015,0.0052,0.0017
3,CIC-IDS2017,UNSW-NB15,MLP,intra,0.3125,0.7693,0.6854,-0.0839,0.6972,0.9138,0.8841,0.5685,0.2577,0.1294
4,CIC-IDS2017,UNSW-NB15,Random Forest,cross,0.6889,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
5,CIC-IDS2017,UNSW-NB15,Random Forest,intra,0.6889,0.9536,0.9513,-0.0024,0.9402,0.9284,0.9687,0.9765,0.2043,0.2086
6,UNSW-NB15,CIC-IDS2017,Logistic Regression,cross,0.6951,0.4474,0.3321,-0.1153,0.3506,0.1995,0.7109,0.9885,0.4366,0.9803
7,UNSW-NB15,CIC-IDS2017,Logistic Regression,intra,0.6951,0.4261,0.3642,-0.0619,0.3705,0.2305,0.5170,0.8687,0.1591,0.4737
8,UNSW-NB15,CIC-IDS2017,MLP,cross,0.7381,0.1385,0.1611,0.0227,0.1526,0.1551,0.1317,0.1696,0.1539,0.2201
9,UNSW-NB15,CIC-IDS2017,MLP,intra,0.7381,0.6066,0.9133,0.3067,0.8831,0.9009,0.5558,0.9299,0.0774,0.1300


## 8. Analyse des écarts par expérience

Cette vue permet d'identifier si l'effet du seuil est homogène ou concentré sur certaines expériences. Les écarts positifs signifient que le seuil 0,5 améliore la métrique concernée par rapport au seuil sélectionné ; les écarts négatifs signifient l'inverse.

In [8]:
display(
    ablation[
        [
            "source", "target", "model", "fold", "seed", "evaluation",
            "selected_threshold",
            "selected_f1", "fixed_f1", "delta_f1_fixed_minus_selected",
            "selected_alert_rate", "fixed_alert_rate",
            "validation_f1_selected_threshold", "validation_f1_fixed_05"
        ]
    ]
    .sort_values(
        ["source", "target", "model", "evaluation", "delta_f1_fixed_minus_selected"]
    )
    .round(4)
)

,source,target,model,fold,seed,evaluation,selected_threshold,selected_f1,fixed_f1,delta_f1_fixed_minus_selected,selected_alert_rate,fixed_alert_rate,validation_f1_selected_threshold,validation_f1_fixed_05
23,CIC-IDS2017,UNSW-NB15,Logistic Regression,4,2026,cross,0.4802,0.7933,0.0023,-0.7909,0.0935,0.0037,0.6171,0.5620
3,CIC-IDS2017,UNSW-NB15,Logistic Regression,1,123,cross,0.4763,0.0005,0.0005,0.0000,0.0027,0.0026,0.6228,0.3947
11,CIC-IDS2017,UNSW-NB15,Logistic Regression,2,2026,cross,0.4749,0.0005,0.0005,0.0000,0.0027,0.0026,0.6681,0.4071
25,CIC-IDS2017,UNSW-NB15,Logistic Regression,5,42,cross,0.4758,0.0005,0.0005,0.0000,0.0027,0.0026,0.6401,0.4102
27,CIC-IDS2017,UNSW-NB15,Logistic Regression,5,123,cross,0.4774,0.0005,0.0005,0.0000,0.0027,0.0026,0.6213,0.4000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98,UNSW-NB15,CIC-IDS2017,Random Forest,2,123,intra,0.9868,0.2273,0.9446,0.7173,0.0167,0.1279,0.9998,0.9856
96,UNSW-NB15,CIC-IDS2017,Random Forest,2,42,intra,0.9837,0.2196,0.9438,0.7243,0.0158,0.1279,0.9994,0.9833
112,UNSW-NB15,CIC-IDS2017,Random Forest,4,2026,intra,0.9850,0.1470,0.9400,0.7930,0.0100,0.1248,0.9996,0.9866
108,UNSW-NB15,CIC-IDS2017,Random Forest,4,42,intra,0.9999,0.0778,0.9535,0.8757,0.0051,0.1277,0.9996,0.9843


## 9. Sauvegarde des résultats

Les résultats détaillés et les tableaux synthétiques sont enregistrés dans `results/tables/`. Aucun fichier du Notebook 04 n'est écrasé.

In [9]:
DETAIL_FILE = TABLES_DIR / "threshold_ablation_v1_1.csv"
SUMMARY_FILE = TABLES_DIR / "threshold_ablation_summary_v1_1.csv"
COMPACT_FILE = TABLES_DIR / "threshold_ablation_compact_v1_1.csv"

ablation.to_csv(DETAIL_FILE, index=False)
summary.to_csv(SUMMARY_FILE)
compact.to_csv(COMPACT_FILE, index=False)

print("Résultats détaillés :", DETAIL_FILE)
print("Résumé moyen/std    :", SUMMARY_FILE)
print("Résumé compact      :", COMPACT_FILE)
print("Nombre d'expériences source :", len(results))
print("Nombre de lignes détaillées :", len(ablation))

Résultats détaillés : D:\Master-SDIA\CYB-1-IDS-Generalization\results\tables\threshold_ablation_v1_1.csv
Résumé moyen/std    : D:\Master-SDIA\CYB-1-IDS-Generalization\results\tables\threshold_ablation_summary_v1_1.csv
Résumé compact      : D:\Master-SDIA\CYB-1-IDS-Generalization\results\tables\threshold_ablation_compact_v1_1.csv
Nombre d'expériences source : 90
Nombre de lignes détaillées : 180


In [10]:
# --- Diagnostic : pourquoi le seuil sélectionné sur UNSW échoue-t-il ? ---
DATA_DIR = ROOT_DIR / "data" / "interim"
FEATURES = ["flow_duration_s", "fwd_packets", "bwd_packets", "fwd_bytes", "bwd_bytes"]

def load_groups(filename):
    X = pd.read_csv(DATA_DIR / filename, usecols=FEATURES)
    g, _ = pd.factorize(pd.MultiIndex.from_frame(X), sort=False)
    return np.asarray(g, dtype=np.int64)

GROUPS = {"CIC-IDS2017": load_groups("cic_harmonized_sample_v2.csv"),
          "UNSW-NB15": load_groups("unsw_harmonized_sample_v2.csv")}

rows = []
for r in results.itertuples(index=False):
    s_short = "CIC" if r.source == "CIC-IDS2017" else "UNSW"
    t_short = "CIC" if r.target == "CIC-IDS2017" else "UNSW"
    with np.load(PRED_DIR / prediction_filename(s_short, t_short, r.model, int(r.fold), int(r.seed))) as p:
        thr = float(p["threshold"]); vi = p["inner_val_idx"]
        yv, sv = p["inner_val_y_true"].astype(int), p["inner_val_score"]
        yt, st = p["source_test_y_true"].astype(int), p["source_test_score"]
    g = GROUPS[r.source][vi]
    att = g[yv == 1]
    rows.append({
        "source": r.source, "model": r.model, "fold": int(r.fold), "seed": int(r.seed), "seuil": thr,
        "part_val_dans_plus_gros_groupe": np.bincount(g).max() / len(g),
        "part_attaques_val_dans_plus_gros_groupe": np.bincount(att).max() / len(att) if len(att) else np.nan,
        "rappel_validation": ((sv >= thr) & (yv == 1)).sum() / max((yv == 1).sum(), 1),
        "rappel_test_externe": ((st >= thr) & (yt == 1)).sum() / max((yt == 1).sum(), 1),
    })
diag = pd.DataFrame(rows)
display(diag.groupby(["source", "model"]).mean(numeric_only=True).drop(columns=["fold", "seed"]).round(4))
display(diag[(diag.source == "UNSW-NB15") & (diag.model == "Random Forest")]
        .sort_values("seuil", ascending=False).round(4))
diag.to_csv(TABLES_DIR / "threshold_failure_diagnostic_v1_1.csv", index=False)

seuil  part_val_dans_plus_gros_groupe  part_attaques_val_dans_plus_gros_groupe  rappel_validation  rappel_test_externe
source      model                                                                                                                                       
CIC-IDS2017 Logistic Regression  0.4742                          0.0428                                   0.0663             0.8113               0.7871
            MLP                  0.3125                          0.0428                                   0.0663             0.9130               0.8841
            Random Forest        0.6889                          0.0428                                   0.0663             0.9701               0.9687
UNSW-NB15   Logistic Regression  0.6951                          0.1355                                   0.7215             0.9153               0.5170
            MLP                  0.7381                          0.1355                                   0.7215             0.9820               0.5558
            Random Forest        0.7257                          0.1355                                   0.7215             0.9896               0.5246

,source,model,fold,seed,seuil,part_val_dans_plus_gros_groupe,part_attaques_val_dans_plus_gros_groupe,rappel_validation,rappel_test_externe
55,UNSW-NB15,Random Forest,4,123,1.0000,0.1501,1.0000,1.0000,0.0317
54,UNSW-NB15,Random Forest,4,42,0.9999,0.1501,1.0000,1.0000,0.0405
59,UNSW-NB15,Random Forest,5,2026,0.9889,0.1501,1.0000,1.0000,0.1767
49,UNSW-NB15,Random Forest,2,123,0.9868,0.1501,1.0000,1.0000,0.1287
56,UNSW-NB15,Random Forest,4,2026,0.9850,0.1501,1.0000,1.0000,0.0794
48,UNSW-NB15,Random Forest,2,42,0.9837,0.1501,1.0000,1.0000,0.1236
58,UNSW-NB15,Random Forest,5,123,0.9772,0.1501,1.0000,1.0000,0.6104
57,UNSW-NB15,Random Forest,5,42,0.9721,0.1501,1.0000,1.0000,0.6192
50,UNSW-NB15,Random Forest,2,2026,0.9488,0.1501,1.0000,1.0000,0.1740
53,UNSW-NB15,Random Forest,3,2026,0.3650,0.1142,0.3131,0.9684,0.9630
